In [1]:
LAST_NAME = "Gabilan"
STUDENT_ID = "TUPM-26-2636"

SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(digit) for digit in STUDENT_ID if digit.isdigit())
NAME_LENGTH = len(LAST_NAME)

TARGET_YEAR = 2015 + SEED_DIGIT

print("Student:", LAST_NAME)
print("Student ID:", STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Name Length:", NAME_LENGTH)
print("Target Year:", TARGET_YEAR)

Student: Gabilan
Student ID: TUPM-26-2636
Seed Digit: 6
ID Sum: 25
Name Length: 7
Target Year: 2021


In [2]:
import csv
import json
import os
import shutil
from collections import Counter

In [3]:
INPUT_FILE = "Global_Cybersecurity_Threats_2015-2024.csv"

OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FILTERED_FILE = os.path.join(OUTPUT_DIR, "filtered_cybersecurity_incidents.csv")
SUMMARY_FILE = os.path.join(OUTPUT_DIR, "cybersecurity_summary.json")
BACKUP_FILE = os.path.join(OUTPUT_DIR, "cybersecurity_summary_backup.json")
REPORT_FILE = os.path.join(OUTPUT_DIR, "incident_report.txt")

print("Input File:", INPUT_FILE)
print("Target Year:", TARGET_YEAR)
print("Output Directory:", OUTPUT_DIR)

Input File: Global_Cybersecurity_Threats_2015-2024.csv
Target Year: 2021
Output Directory: output


In [4]:
total_records = 0
target_records = []
invalid_years = 0
invalid_losses = 0

with open(INPUT_FILE, "r", newline="", encoding="utf-8-sig") as file:
    reader = csv.DictReader(file)
    fieldnames = reader.fieldnames

    for row in reader:
        total_records += 1

        try:
            year = int(row["Year"])

            if year != TARGET_YEAR:
                continue

            try:
                loss = float(row["Financial Loss (in Million $)"])

                if loss < 0:
                    raise ValueError

            except (ValueError, TypeError):
                loss = None
                invalid_losses += 1

            row["_loss_numeric"] = loss
            target_records.append(row)

        except (KeyError, ValueError, TypeError):
            invalid_years += 1

print("Total Dataset Records:", total_records)
print("Incidents in Target Year:", len(target_records))
print("Invalid Year Records:", invalid_years)
print("Invalid Financial Values:", invalid_losses)

Total Dataset Records: 3000
Incidents in Target Year: 299
Invalid Year Records: 0
Invalid Financial Values: 0


In [5]:
attack_counts = Counter(
    (record.get("Attack Type") or "Unknown")
    for record in target_records
)

print("Incident Count per Attack Type:")

for attack_type, count in sorted(attack_counts.items()):
    print(attack_type, ":", count)

Incident Count per Attack Type:
DDoS : 47
Malware : 49
Man-in-the-Middle : 48
Phishing : 41
Ransomware : 55
SQL Injection : 59


In [6]:
valid_losses = [
    record["_loss_numeric"]
    for record in target_records
    if record["_loss_numeric"] is not None
]

if valid_losses:
    total_loss = sum(valid_losses)
    average_loss = total_loss / len(valid_losses)
else:
    total_loss = 0
    average_loss = None

print("Total Financial Loss:", total_loss, "Million USD")
print("Average Financial Loss:", average_loss, "Million USD")
print("Valid Financial Loss Records:", len(valid_losses))

Total Financial Loss: 15873.41 Million USD
Average Financial Loss: 53.08832775919733 Million USD
Valid Financial Loss Records: 299


In [7]:
ranked_incidents = sorted(
    enumerate(target_records),
    key=lambda item: (
        -(item[1]["_loss_numeric"])
        if item[1]["_loss_numeric"] is not None
        else float("inf"),
        item[0]
    )
)

top_five = [
    record for index, record in ranked_incidents
    if record["_loss_numeric"] is not None
][:5]

print("===== FIVE COSTLIEST INCIDENTS =====")

for rank, record in enumerate(top_five, start=1):
    print("\nRank:", rank)
    for key, value in record.items():
        if key != "_loss_numeric":
            print(key + ":", value)

===== FIVE COSTLIEST INCIDENTS =====

Rank: 1
Country: USA
Year: 2021
Attack Type: DDoS
Target Industry: Telecommunications
Financial Loss (in Million $): 99.81
Number of Affected Users: 460842
Attack Source: Unknown
Security Vulnerability Type: Weak Passwords
Defense Mechanism Used: AI-based Detection
Incident Resolution Time (in Hours): 59

Rank: 2
Country: France
Year: 2021
Attack Type: Ransomware
Target Industry: IT
Financial Loss (in Million $): 99.77
Number of Affected Users: 533151
Attack Source: Nation-state
Security Vulnerability Type: Zero-day
Defense Mechanism Used: Antivirus
Incident Resolution Time (in Hours): 43

Rank: 3
Country: France
Year: 2021
Attack Type: SQL Injection
Target Industry: Government
Financial Loss (in Million $): 98.76
Number of Affected Users: 904598
Attack Source: Hacker Group
Security Vulnerability Type: Social Engineering
Defense Mechanism Used: AI-based Detection
Incident Resolution Time (in Hours): 11

Rank: 4
Country: China
Year: 2021
Attack Type

In [8]:
with open(FILTERED_FILE, "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)

    writer.writeheader()

    for record in target_records:
        clean_record = {
            key: value for key, value in record.items()
            if key != "_loss_numeric"
        }
        writer.writerow(clean_record)

print("Filtered CSV saved:", FILTERED_FILE)

Filtered CSV saved: output/filtered_cybersecurity_incidents.csv


In [9]:
clean_top_five = []

for record in top_five:
    clean_top_five.append({
        key: value for key, value in record.items()
        if key != "_loss_numeric"
    })

summary = {
    "Student": LAST_NAME,
    "Student ID": STUDENT_ID,
    "Target Year": TARGET_YEAR,
    "Total Records in Dataset": total_records,
    "Incidents in Target Year": len(target_records),
    "Count per Attack Type": dict(attack_counts),
    "Total Financial Loss (USD Millions)": total_loss,
    "Average Financial Loss (USD Millions)": average_loss,
    "Valid Financial Loss Records": len(valid_losses),
    "Five Costliest Incidents": clean_top_five,
    "Invalid Year Records": invalid_years,
    "Invalid Financial Values": invalid_losses
}

with open(SUMMARY_FILE, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

print("JSON Summary saved:", SUMMARY_FILE)

JSON Summary saved: output/cybersecurity_summary.json


In [10]:
shutil.copy2(SUMMARY_FILE, BACKUP_FILE)

backup_verified = (
    os.path.exists(BACKUP_FILE)
    and os.path.getsize(BACKUP_FILE) == os.path.getsize(SUMMARY_FILE)
)

print("Backup Created:", os.path.exists(BACKUP_FILE))
print("Backup Verified:", backup_verified)

if backup_verified:
    print("Backup File Size:", os.path.getsize(BACKUP_FILE), "bytes")

Backup Created: True
Backup Verified: True
Backup File Size: 3018 bytes


In [11]:
with open(REPORT_FILE, "w", encoding="utf-8") as file:
    file.write("CYBERSECURITY INCIDENT REPORT\n")
    file.write("=" * 40 + "\n")

    file.write(f"Student: {LAST_NAME}\n")
    file.write(f"Student ID: {STUDENT_ID}\n")
    file.write(f"Target Year: {TARGET_YEAR}\n")
    file.write(f"Total Dataset Records: {total_records}\n")
    file.write(f"Incidents in Target Year: {len(target_records)}\n")

    file.write("\nINCIDENT COUNT PER ATTACK TYPE\n")

    for attack_type, count in sorted(attack_counts.items()):
        file.write(f"{attack_type}: {count}\n")

    file.write("\nFINANCIAL LOSS SUMMARY\n")
    file.write(f"Total Financial Loss: {total_loss} Million USD\n")
    file.write(f"Average Financial Loss: {average_loss} Million USD\n")

    file.write("\nFIVE COSTLIEST INCIDENTS\n")

    for rank, record in enumerate(top_five, start=1):
        file.write(f"\nRank {rank}\n")

        for key, value in record.items():
            if key != "_loss_numeric":
                file.write(f"{key}: {value}\n")

    file.write("\nOUTPUT FILES\n")
    file.write(f"Filtered CSV: {FILTERED_FILE}\n")
    file.write(f"JSON Summary: {SUMMARY_FILE}\n")
    file.write(f"Backup: {BACKUP_FILE}\n")
    file.write(f"Backup Verified: {backup_verified}\n")
    file.write(f"Report: {REPORT_FILE}\n")

print("Text Report Generated:", REPORT_FILE)

Text Report Generated: output/incident_report.txt


In [12]:
print("===== CYBERSECURITY INCIDENT REPORT =====")
print("Target Year:", TARGET_YEAR)
print("Total Dataset Records:", total_records)
print("Incidents in Target Year:", len(target_records))

print("\nCount per Attack Type:")
for attack_type, count in sorted(attack_counts.items()):
    print(attack_type, ":", count)

print("\nTotal Financial Loss:", total_loss, "Million USD")
print("Average Financial Loss:", average_loss, "Million USD")

print("\nFive Costliest Incidents:", len(top_five))
print("Filtered CSV:", FILTERED_FILE)
print("JSON Summary:", SUMMARY_FILE)
print("Backup Verified:", backup_verified)
print("Report File:", REPORT_FILE)

print("=========================================")

===== CYBERSECURITY INCIDENT REPORT =====
Target Year: 2021
Total Dataset Records: 3000
Incidents in Target Year: 299

Count per Attack Type:
DDoS : 47
Malware : 49
Man-in-the-Middle : 48
Phishing : 41
Ransomware : 55
SQL Injection : 59

Total Financial Loss: 15873.41 Million USD
Average Financial Loss: 53.08832775919733 Million USD

Five Costliest Incidents: 5
Filtered CSV: output/filtered_cybersecurity_incidents.csv
JSON Summary: output/cybersecurity_summary.json
Backup Verified: True
Report File: output/incident_report.txt
